# Sesion 7 - Workflows inteligentes y orquestacion

Este notebook separa el laboratorio en tres capas: modelo predictivo, simulacion determinista y workflow multiagente.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from agents.intelligent_workflow import RandomForestConfig, WorkflowRequest
from agents.intelligent_workflow.analytics import load_campaign_data, train_discount_model, simulate_discount_scenarios
from agents.intelligent_workflow.workflow import IntelligentWorkflow

## 1. Datos y modelo configurable

Modifique profundidad, numero de arboles o tamano de prueba. La semilla mantiene el ejercicio reproducible.

In [ ]:
data = load_campaign_data()
config = RandomForestConfig(n_estimators=200, max_depth=10, min_samples_leaf=2, test_size=0.2)
bundle = train_discount_model(data, config)
bundle.metrics.model_dump()

## 2. Escenarios deterministas

El modelo predice uplift. Las formulas de negocio calculan pedidos, ingresos y margen sin usar el LLM.

In [ ]:
scenarios = simulate_discount_scenarios(data, bundle, 'Pereira', 'Alto valor', [0, 5, 10, 15, 20])
[scenario.model_dump() for scenario in scenarios]

## 3. Workflow multiagente

Esta celda requiere Ollama y `qwen2.5:3b`. Observe los tres especialistas, sus fuentes y la traza del Reviewer.

In [ ]:
request = WorkflowRequest(
    question='Que descuento debe probar NovaRetail sin deteriorar su margen?',
    city='Pereira',
    segment='Alto valor',
    forest=config,
)
result = IntelligentWorkflow().run(request, model_bundle=bundle)
print(result.report_markdown)

In [ ]:
for finding in result.findings:
    print(f'{finding.agent_name}: {finding.status} ({finding.elapsed_seconds:.2f}s)')
    print(finding.summary, '\n')

[event.model_dump(mode='json') for event in result.trace]

## 4. Pregunta de seguimiento

El chat usa el resultado ya terminado. No reentrena el modelo ni modifica la recomendacion.

In [ ]:
answer = IntelligentWorkflow().answer_follow_up(result, 'Por que no conviene el descuento mas alto?')
answer.model_dump()